In [1]:
!pip install kagglehub

import kagglehub

# Correct dataset: grassknoted/asl-alphabet
path = kagglehub.dataset_download("grassknoted/asl-alphabet")

print("Dataset Downloaded to:", path)

Using Colab cache for faster access to the 'asl-alphabet' dataset.
Dataset Downloaded to: /kaggle/input/asl-alphabet


In [5]:
import os

DATASET_PATH = "/kaggle/input/asl-alphabet/asl_alphabet_train/asl_alphabet_train"
if os.path.exists(DATASET_PATH):
    classes = sorted(os.listdir(DATASET_PATH))
    print(f"Total Classes: {len(classes)}")
    print("Classes:", classes)

Total Classes: 29
Classes: ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'del', 'nothing', 'space']


In [6]:
import os
import cv2
import numpy as np
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

# Download MediaPipe Hand Landmarker model bundle
!wget -q -O hand_landmarker.task https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task

# Setup Task Landmarker options
base_options = python.BaseOptions(model_asset_path='hand_landmarker.task')
options = vision.HandLandmarkerOptions(base_options=base_options, num_hands=1)
detector = vision.HandLandmarker.create_from_options(options)

DATASET_PATH = "/kaggle/input/asl-alphabet/asl_alphabet_train/asl_alphabet_train"
classes = sorted(os.listdir(DATASET_PATH))

X_data = []
y_data = []
IMAGES_PER_CLASS = 500  # Fast processing සඳහා class එකකට images 500ක්

print("Extracting Landmarks...")

for class_name in classes:
    class_dir = os.path.join(DATASET_PATH, class_name)
    image_files = os.listdir(class_dir)[:IMAGES_PER_CLASS]
    print(f"Processing Class: {class_name}")

    for img_name in image_files:
        img_path = os.path.join(class_dir, img_name)

        # Load image via MediaPipe Image object
        mp_image = mp.Image.create_from_file(img_path)
        detection_result = detector.detect(mp_image)

        if detection_result.hand_landmarks:
            hand_landmarks = detection_result.hand_landmarks[0]

            # Wrist Relative Normalization (Point 0 relative)
            wrist_x = hand_landmarks[0].x
            wrist_y = hand_landmarks[0].y
            wrist_z = hand_landmarks[0].z

            landmarks = []
            for lm in hand_landmarks:
                landmarks.extend([
                    lm.x - wrist_x,
                    lm.y - wrist_y,
                    lm.z - wrist_z
                ])

            X_data.append(landmarks)
            y_data.append(class_name)

X_data = np.array(X_data)
y_data = np.array(y_data)

print(f"\nExtraction Completed Successfully!")
print(f"Total Extracted Samples: {X_data.shape[0]}")

Extracting Landmarks...
Processing Class: A
Processing Class: B
Processing Class: C
Processing Class: D
Processing Class: E
Processing Class: F
Processing Class: G
Processing Class: H
Processing Class: I
Processing Class: J
Processing Class: K
Processing Class: L
Processing Class: M
Processing Class: N
Processing Class: O
Processing Class: P
Processing Class: Q
Processing Class: R
Processing Class: S
Processing Class: T
Processing Class: U
Processing Class: V
Processing Class: W
Processing Class: X
Processing Class: Y
Processing Class: Z
Processing Class: del
Processing Class: nothing
Processing Class: space

Extraction Completed Successfully!
Total Extracted Samples: 10615


In [7]:
import pandas as pd

# Save features and labels as CSV file
df = pd.DataFrame(X_data)
df['label'] = y_data
df.to_csv("asl_hand_landmarks.csv", index=False)

print("asl_hand_landmarks.csv file saved successfully!")

asl_hand_landmarks.csv file saved successfully!
